In [1]:
import os

def setting_up_proxy(proxy=None, proxy_type='http', verbose=True):
    supported_proxy_types = ['http', 'https', 'socks4', 'socks5', 'all']
    assert proxy_type in supported_proxy_types, f"proxy type {repr(proxy_type)} not supported, only support {supported_proxy_types}"
    if proxy is None:
        proxy = os.environ.get(f'{proxy_type}_proxy')
    if proxy is None:
        return
    if verbose:
        print(f'setting up proxy {repr(proxy)} for {repr(proxy_type)}')
    os.environ[f'{proxy_type}_proxy'] = proxy


# default_proxy_config = {
#     'http': 'http://127.0.0.1:7890',
#     'https': 'http://127.0.0.1:7890',
#     'all': 'socks5://127.0.0.1:7890',
# }


default_proxy_config = {
    'http': 'http://10.176.52.116:7890',
    'https': 'http://10.176.52.116:7890',
    'all': 'socks5://10.176.52.116:7890',
}


def setting_up_proxy_from_config(proxy_config=default_proxy_config, verbose=True):
    for proxy_type, proxy_url in proxy_config.items():
        setting_up_proxy(proxy=proxy_url, proxy_type=proxy_type, verbose=verbose)
    print()


# setting_up_proxy_from_config()

In [2]:
import requests
import os
import json
resp = requests.get('https://www.xmwav.com/mscdetail/133438.html', proxies=default_proxy_config)
print(resp)
print(resp.headers['content-type'])

with open('examples/133438.html', 'wb') as f:
    f.write(resp.content)

<Response [200]>
text/html; charset=utf-8


In [8]:
import re
from RFC.utils.parse import (
    get_html_soup,
    parse,
)


def parse_music_page(resp):
    parse_config = {
        'title': {
            ('attr', 'div', 'class', 'info-bt', None): {
                ('attr', 'h1', 'class', 'title', None): {
                    ('result', 'text', (('return_as_list', False),), None): {},
                },
                ('type', 'h2', 0): {
                    ('result', 'text', (('return_as_list', False),), None): {},
                }
            }
        },
        'metadata': {
            ('attr', 'div', 'class', 'info-bt', None): {
                ('type', 'small', 0): {
                    ('result', 'text', (('return_as_list', True),), None): {},
                }
            }
        },
        'tags': {
            ('attr', 'div', 'class', 'info-bt', None): {
                ('type', 'h5', 0): {
                    ('result', 'text', (('return_as_list', False),), None): {},
                }
            }
        },
        'links_title': {
            ('attr', 'div', 'class', 'info-zi mb15', None): {
                ('result', 'text', (('return_as_list', True),), None): {},
            }
        },
        'links': {
            ('attr', 'div', 'class', 'info-zi mb15', None): {
                ('result', 'href', None): {},
            }
        },
        'lyrics': {
            ('attr', 'div', 'class', 'lrc', None): {
                ('type', 'article', None): {
                    ('result', 'text', (('return_as_list', True),), None): {},
                }
            }
        },
    }
    title = parse(get_html_soup(resp.content), parse_config['title'])
    metadata = parse(get_html_soup(resp.content), parse_config['metadata'])
    tags = parse(get_html_soup(resp.content), parse_config['tags'])
    links_title = parse(get_html_soup(resp.content), parse_config['links_title'])
    links = parse(get_html_soup(resp.content), parse_config['links'])
    lyrics = parse(get_html_soup(resp.content), parse_config['lyrics'])
    return {
        'title': title,
        'metadata': metadata,
        'tags': tags,
        'links_title': links_title,
        'links': links,
        'lyrics': lyrics,
    }
    

# def parse_music_page_re(resp):
#     info = {k: v for k, v in re.findall('window.([0-9a-z_]*) = (.*);', resp.text)}
#     info['mp3_lrc'] = [part for part in '\n'.join(re.findall('window.mp3_lrc = ((?:.*\n)*)`;', resp.text, re.MULTILINE)).split('\n') if part]
#     return info

info = parse_music_page(resp)
# print(info)
import json
print(json.dumps(info, indent=4, ensure_ascii=False))

{
    "title": [
        "《铃芽之旅》周深mp3歌曲下载",
        "[WAV/MP3]-40M"
    ],
    "metadata": [
        [
            "分享时间：2023-03-2112:09:47",
            "热度：12981"
        ]
    ],
    "tags": [
        "音乐标签：翻唱,网红热门,动漫,新歌推荐"
    ],
    "links_title": [
        [
            "夸克WAV链接下载",
            "夸克MP3链接下载",
            "迅雷备用链接"
        ]
    ],
    "links": [
        "https://www.xmwav.com/download/133438.html",
        "https://www.xmwav.com/download_m/m133438.html",
        "https://www.xmwav.com/download_xl/xl133438.html"
    ],
    "lyrics": [
        [],
        [
            "﻿你身体中存在的红蓝相交的线",
            "它们最终于心脏之中交织汇合",
            "如今正利用于风中也不会减弱的声音",
            "于我心中孕育着渴望传递给你的话语",
            "时间枕木悄逝微风拂过嫩肤星辰众魂故土人如蜉蝣虚无",
            "你问我为何在哭泪水即为我的答复",
            "你我有幸相遇的意义仿佛踪迹全无",
            "孤身一人的嘶吼还远远不够",
            "唯有你的触碰才能使我的心脏开始跳动",
            "究竟要跨越多少意义你我才能抵达目的地",
            "愚蠢也好丑陋也罢我只愿在那正确的方向与你携手前行",
            "无法回想起重要的记忆",
            "心中想法难以化为言语",
      